# 03 — Export ONNX, upload to S3, register in OpenShift AI Model Registry

Only the **selected** MLflow run becomes a registered model version.

Flow: `SELECTED_RUN_ID` → export ONNX → versioned S3 key → Model Registry (`stage=candidate`).

Requires S3 + `MODEL_REGISTRY_URL` (see `docs/env.md`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import mlflow
from mlflow.tracking import MlflowClient
from ultralytics import YOLO

from src.mlflow_utils import configure_tracking
from src.registry_client import ModelRegistryClient
from src.s3_utils import storage_path_from_uri, upload_file, versioned_key

configure_tracking()
client = MlflowClient()

run_id = os.environ.get("SELECTED_RUN_ID")
if not run_id:
    raise SystemExit("Set SELECTED_RUN_ID (from notebooks/02_evaluate_select.ipynb)")

run = client.get_run(run_id)
run_name = run.data.params.get("run_name") or run.info.run_name or run_id[:8]
local_weights = run.data.tags.get("weights_local_path") or f"runs-openshift/{run_name}/weights/best.pt"
print("run_id:", run_id)
print("run_name:", run_name)
print("weights:", local_weights)
assert Path(local_weights).is_file(), f"Missing weights: {local_weights}"

In [ ]:
model = YOLO(local_weights)
onnx_path = Path(model.export(format="onnx"))
# Ultralytics writes best.onnx next to best.pt
if not onnx_path.is_file():
    onnx_path = Path(local_weights).with_name("best.onnx")
print("ONNX:", onnx_path)

In [ ]:
key = versioned_key(run_id)
s3_uri = upload_file(onnx_path, key)
print("Uploaded:", s3_uri)

with mlflow.start_run(run_id=run_id):
    mlflow.log_param("onnx_s3_uri", s3_uri)
    mlflow.set_tag("onnx_s3_key", key)
    if onnx_path.is_file():
        mlflow.log_artifact(str(onnx_path), artifact_path="onnx")

In [ ]:
model_name = os.environ.get("REGISTERED_MODEL_NAME", "xray-detector")
metrics = {
    k.replace("metrics_", "").replace("metrics/", ""): v
    for k, v in run.data.metrics.items()
    if isinstance(v, (int, float))
}

registry = ModelRegistryClient()
result = registry.register_onnx_model(
    model_name=model_name,
    version_name=f"from-mlflow-{run_id[:8]}",
    s3_uri=s3_uri,
    mlflow_run_id=run_id,
    metrics=metrics,
    storage_key=os.environ.get("STORAGE_KEY"),
    storage_path=storage_path_from_uri(s3_uri),
    description=f"Curated from MLflow run {run_id} ({run_name})",
)

version_id = str(result["model_version"]["id"])
os.environ["SELECTED_VERSION_ID"] = version_id
print("Registered model:", model_name)
print("Version id:", version_id)
print("Version name:", result["model_version"].get("name"))
print("stage=candidate (promote in notebook 04)")
print("SELECTED_VERSION_ID=" + version_id)